# Preparación e Integración de Datos

## Objetivo

Este notebook realiza el proceso completo de limpieza, transformación e integración de los datasets utilizados para el análisis y modelado predictivo de precios agrícolas.

El resultado final es un único conjunto de datos consistente, listo para ser utilizado en análisis exploratorio, entrenamiento de modelos y visualización.

---

## Flujo del proceso

1. Carga de los datos agrícolas provenientes de múltiples archivos anuales.
2. Limpieza y estandarización de variables.
3. Corrección de valores faltantes mediante reglas derivadas de la estructura de los datos.
4. Consolidación de todos los años en un único DataFrame.
5. Carga y procesamiento de los datos meteorológicos diarios.
6. Agregación de información meteorológica a frecuencia semanal.
7. Integración de ambas fuentes mediante región, año y semana ISO.
8. Exportación del dataset final.

## Diccionario de datos: 

| Variable | Tipo | Descripción |
|----------|------|-------------|
| **semana** | Entero | Número de semana del año (calendario ISO) correspondiente al período de observación. |
| **fecha_inicio** | Fecha | Fecha de inicio de la semana de monitoreo. |
| **fecha_termino** | Fecha | Fecha de término de la semana de monitoreo. |
| **region** | Categórica | Región de Chile donde se realizó el monitoreo de precios. |
| **tipo_punto_monitoreo** | Categórica | Tipo de establecimiento donde se registró el precio (por ejemplo: feria libre, supermercado, mercado mayorista, entre otros). |
| **producto** | Categórica | Nombre del producto agrícola monitoreado. |
| **variedad** | Categórica | Variedad específica del producto, cuando corresponde. |
| **calidad** | Categórica | Clasificación de calidad informada para el producto. |
| **unidad** | Categórica | Unidad de medida utilizada para registrar el precio (kg, unidad, bandeja, etc.). |
| **precio_minimo** | Numérica | Precio mínimo observado durante la semana para el producto. |
| **precio_maximo** | Numérica | Precio máximo observado durante la semana para el producto. |
| **precio_promedio** | Numérica | Precio promedio observado durante la semana para el producto. |

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#
from src.preprocessing import formatCategorical, formatColNames

# Data Agrícola

## Preparación de los datos agrícolas

Los datos originales provienen de múltiples archivos obtenidos mediante consultas al sistema de la oficina de estudios y políticas agrarias(ODEPA). Los archivos presentan formatos con columnas auxiliares generadas por Excel y valores faltantes en algunas variables categóricas que deben ser corregidas.

En esta sección se realiza la limpieza y normalización necesaria para obtener un único dataset homogéneo.



In [185]:
#Los archivos no son limpios por lo que dará un warning sobre su formato.
#Aspectos estéticos del archivo hacen que el nombre de columnas se obtenda de la fila 4
df_by_year = {
       '20-21':  pd.read_excel(
            "./Data/precio-consumidor_semanal_202001-202151.xlsx",
            header=4
        ),
        '21-22':  pd.read_excel(
            "./Data/precio-consumidor_semanal_202152-202252.xlsx",
            header=4
        ),
        '23-24': pd.read_excel(
            "./Data/precio-consumidor_semanal_202301-202401.xlsx",
            header=4
        ),
        '24-25': pd.read_excel(
            "./Data/precio-consumidor_semanal_202402-202501.xlsx",
            header=4
        ),
        '25-26':pd.read_excel(
            "./Data/precio-consumidor_semanal_202502-202627.xlsx",
            header=4
        )
}

C:\Users\Joaquin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\Joaquin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\Joaquin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\Joaquin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply o

In [193]:
df_by_year['20-21'].head()

,semana,fecha_inicio,region,tipo_punto_monitoreo,producto,variedad,calidad,unidad,precio_minimo,precio_maximo,precio_promedio
0,1,30-12-2019,región_de_arica_y_parinacota,mercado_minorista,aceituna_amarga_(extra),NaN,NaN,$/kilo,3500.0,4000.0,3750.0
1,1,30-12-2019,región_de_arica_y_parinacota,mercado_minorista,aceituna_amarga_(primera),NaN,NaN,$/kilo,3000.0,3500.0,3250.0
2,1,30-12-2019,región_de_arica_y_parinacota,mercado_minorista,aceituna_amarga_(segunda),NaN,NaN,$/kilo,2500.0,3000.0,2750.0
3,1,30-12-2019,región_de_arica_y_parinacota,mercado_minorista,aceituna_amarga_(tercera),NaN,NaN,$/kilo,2000.0,2500.0,2250.0
4,1,30-12-2019,región_de_arica_y_parinacota,mercado_minorista,aceituna_negra_(extra),NaN,NaN,$/kilo,3000.0,3500.0,3250.0


In [194]:
categorical_cols = list(df_by_year['20-21'].dtypes[df_by_year['20-21'].dtypes == 'str'].index) # usamos el primer dataframe como marco de referencia
categorical_cols 

['fecha_inicio',
 'region',
 'tipo_punto_monitoreo',
 'producto',
 'variedad',
 'calidad',
 'unidad']

In [195]:
#Semana y fecha término son dispensables. seguiremos solo con fecha inicio
categorical_cols.remove("Fecha término")

ValueError: list.remove(x): x not in list

### Limpieza y estandarización

Las principales transformaciones realizadas son:

- eliminación de columnas vacías o auxiliares;
- normalización de nombres de columnas;
- estandarización de variables categóricas;
- conversión de fechas;
- eliminación de registros incompletos cuando corresponde.

In [196]:
def cleanSheets(df):

    #Eliminar columnas estéticos o sobrantes
    df = df.dropna(axis=1, how="all") #los archivos poseen con una columna vacia "unnamed". hay que borrarla
    df = df.drop(columns=[ "Fecha término"])
    #Eliminar última fila
    df = df.iloc[:-1]# Esta fila solo contiene un informativo sobre la fuente de los datos. no es parte de la tabla.
    return df

In [197]:
for year, df in df_by_year.items():
    df_by_year[year] = cleanSheets(df)
    formatCategorical(df_by_year[year], categorical_cols)
    formatColNames(df_by_year[year])

KeyError: "['Fecha término'] not found in axis"

### Consolidación histórica

Una vez limpiados todos los archivos anuales, se consolidan en un único DataFrame que contiene toda la información histórica disponible.

In [198]:
df_agricola = pd.concat(df_by_year.values(), ignore_index=True)

In [199]:
df_agricola

,semana,fecha_inicio,region,tipo_punto_monitoreo,producto,variedad,calidad,unidad,precio_minimo,precio_maximo,precio_promedio
0,1,30-12-2019,región_de_arica_y_parinacota,mercado_minorista,aceituna_amarga_(extra),NaN,NaN,$/kilo,3500.0,4000.0,3750.0
1,1,30-12-2019,región_de_arica_y_parinacota,mercado_minorista,aceituna_amarga_(primera),NaN,NaN,$/kilo,3000.0,3500.0,3250.0
2,1,30-12-2019,región_de_arica_y_parinacota,mercado_minorista,aceituna_amarga_(segunda),NaN,NaN,$/kilo,2500.0,3000.0,2750.0
3,1,30-12-2019,región_de_arica_y_parinacota,mercado_minorista,aceituna_amarga_(tercera),NaN,NaN,$/kilo,2000.0,2500.0,2250.0
4,1,30-12-2019,región_de_arica_y_parinacota,mercado_minorista,aceituna_negra_(extra),NaN,NaN,$/kilo,3000.0,3500.0,3250.0
...,...,...,...,...,...,...,...,...,...,...,...
159259,27,29-06-2026,región_metropolitana_de_santiago,supermercado_en_línea,mango,sin_especificar,primera,$/kilo,2990.0,3010.0,2997.0
159260,27,29-06-2026,región_metropolitana_de_santiago,supermercado_en_línea,manzana,granny_smith,primera,$/kilo,1990.0,2590.0,2415.0
159261,27,29-06-2026,región_metropolitana_de_santiago,supermercado_en_línea,palta,hass,primera,$/kilo,6290.0,6490.0,6423.0
159262,27,29-06-2026,región_metropolitana_de_santiago,supermercado_en_línea,pera,packham's_triumph,primera,$/kilo,2150.0,2190.0,2177.0


### Preparacion previa al futuro Merge con la data meteorológica
    
    -Convertir a fecha el campo string **fecha_inicio**
    -crear el campo **anio** usando isocalendar ( de esta manera la primera semana de 2020, la cual parte el 2019; marcará 2020 como año)
    -Ya que tenemos semana y anio, se creará el campo **mes**. Esta podrá ser eliminada luego si se determina dispensable.


In [200]:
df_agricola["fecha_inicio"] = pd.to_datetime(df_agricola["fecha_inicio"])

iso = df_agricola["fecha_inicio"].dt.isocalendar() # necesito usar ISO por que los datos agrícolas están por semana en este formato.
#ej: 1ra semana de 2020 parte del 30 de diciembre de 2019.

df_agricola["anio"] = iso.year
df_agricola["mes"] = df_agricola["fecha_inicio"].dt.month # Iso no tiene month al parecer..


C:\Users\Joaquin\AppData\Local\Temp\ipykernel_3668\1225151285.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_agricola["fecha_inicio"] = pd.to_datetime(df_agricola["fecha_inicio"])


In [201]:
   df_agricola

,semana,fecha_inicio,region,tipo_punto_monitoreo,producto,variedad,calidad,unidad,precio_minimo,precio_maximo,precio_promedio,anio,mes
0,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna_amarga_(extra),NaN,NaN,$/kilo,3500.0,4000.0,3750.0,2020,12
1,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna_amarga_(primera),NaN,NaN,$/kilo,3000.0,3500.0,3250.0,2020,12
2,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna_amarga_(segunda),NaN,NaN,$/kilo,2500.0,3000.0,2750.0,2020,12
3,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna_amarga_(tercera),NaN,NaN,$/kilo,2000.0,2500.0,2250.0,2020,12
4,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna_negra_(extra),NaN,NaN,$/kilo,3000.0,3500.0,3250.0,2020,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...
159259,27,2026-06-29,región_metropolitana_de_santiago,supermercado_en_línea,mango,sin_especificar,primera,$/kilo,2990.0,3010.0,2997.0,2026,6
159260,27,2026-06-29,región_metropolitana_de_santiago,supermercado_en_línea,manzana,granny_smith,primera,$/kilo,1990.0,2590.0,2415.0,2026,6
159261,27,2026-06-29,región_metropolitana_de_santiago,supermercado_en_línea,palta,hass,primera,$/kilo,6290.0,6490.0,6423.0,2026,6
159262,27,2026-06-29,región_metropolitana_de_santiago,supermercado_en_línea,pera,packham's_triumph,primera,$/kilo,2150.0,2190.0,2177.0,2026,6


In [202]:
df_agricola.isnull().sum()

semana                     0
fecha_inicio               0
region                     0
tipo_punto_monitoreo       0
producto                   0
variedad                3442
calidad                 3442
unidad                     0
precio_minimo              0
precio_maximo              0
precio_promedio            0
anio                       0
mes                        0
dtype: int64

### Tratamiento de valores faltantes

Se detectó que algunos registros no contenían información en las variables **Variedad** y **Calidad**.

Debido a que dicha información se encontraba codificada dentro del nombre del producto, fue posible reconstruir estos valores automáticamente, evitando la pérdida de observaciones.

In [203]:
df_na = df_agricola[df_agricola['variedad'].isna()]

In [204]:
df_na.iloc[1]["producto"].split("_")

['aceituna', 'amarga', '(primera)']

In [205]:
mask = df_na["producto"].str.split("_").str.len() != 3
df_na[mask] # esto confirma que todos los NA salen de informacion en el nombre que debe ser separada

,semana,fecha_inicio,region,tipo_punto_monitoreo,producto,variedad,calidad,unidad,precio_minimo,precio_maximo,precio_promedio,anio,mes


In [206]:
df_agricola[df_agricola["variedad"].isna()]["producto"].str.split("_").str.len().value_counts() # y solo para estar seguros del todo:

producto
3    3442
Name: count, dtype: int64

In [ ]:
#Obtener variedad y calidad a partir del nombre para los casos que haga falta

In [207]:
for row in df_agricola[df_agricola.variedad.isna()].itertuples():
    partes = row.producto.split("_")
    if len(partes) == 3:
        df_agricola.at[row.Index,"producto"] = partes[0]
        df_agricola.at[row.Index,"variedad"] = partes[1]
        df_agricola.at[row.Index,"calidad"] = partes[2]
    

In [208]:
df_agricola.isnull().sum()

semana                  0
fecha_inicio            0
region                  0
tipo_punto_monitoreo    0
producto                0
variedad                0
calidad                 0
unidad                  0
precio_minimo           0
precio_maximo           0
precio_promedio         0
anio                    0
mes                     0
dtype: int64

In [209]:
df_agricola.head()

,semana,fecha_inicio,region,tipo_punto_monitoreo,producto,variedad,calidad,unidad,precio_minimo,precio_maximo,precio_promedio,anio,mes
0,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,amarga,(extra),$/kilo,3500.0,4000.0,3750.0,2020,12
1,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,amarga,(primera),$/kilo,3000.0,3500.0,3250.0,2020,12
2,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,amarga,(segunda),$/kilo,2500.0,3000.0,2750.0,2020,12
3,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,amarga,(tercera),$/kilo,2000.0,2500.0,2250.0,2020,12
4,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,negra,(extra),$/kilo,3000.0,3500.0,3250.0,2020,12


In [210]:
#la columna calidad venia con los valores en parentesis luego del último proceso. hay que quitalos
df_agricola["calidad"] = df_agricola["calidad"].str.replace("(", "", regex=False)
df_agricola["calidad"] = df_agricola["calidad"].str.replace(")", "", regex=False)

df_agricola.head()

,semana,fecha_inicio,region,tipo_punto_monitoreo,producto,variedad,calidad,unidad,precio_minimo,precio_maximo,precio_promedio,anio,mes
0,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,amarga,extra,$/kilo,3500.0,4000.0,3750.0,2020,12
1,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,amarga,primera,$/kilo,3000.0,3500.0,3250.0,2020,12
2,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,amarga,segunda,$/kilo,2500.0,3000.0,2750.0,2020,12
3,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,amarga,tercera,$/kilo,2000.0,2500.0,2250.0,2020,12
4,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,negra,extra,$/kilo,3000.0,3500.0,3250.0,2020,12


In [211]:
#Creamos el archivo nuevo, con toda la data y limpio
#Opto CSV por sobre xslx para que sea visualmente diferenciable incluso sin leer el nombre del archivo.
df_agricola.to_csv("./Data/precio-consumidor-clean.csv", index=False)

# Data meteorológica

## Preparación de los datos meteorológicos

La información meteorológica original corresponde a observaciones diarias.

Como el dataset agrícola se encuentra organizado semanalmente, es necesario transformar ambas fuentes a la misma granularidad temporal antes de integrarlas.

**Más sobre el dataset puede encontrarse en el notebook DataCollection en el cual se recolectó la data**

In [212]:
df_meteo = pd.read_csv("./Data/Meteo.csv")

In [213]:
df_meteo

,time,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,region
0,2020-01-01,23.0,26.2,20.9,0.0,Región de Arica y Parinacota
1,2020-01-02,23.0,26.1,20.4,2.1,Región de Arica y Parinacota
2,2020-01-03,23.6,27.8,20.4,1.0,Región de Arica y Parinacota
3,2020-01-04,23.5,26.4,20.5,0.0,Región de Arica y Parinacota
4,2020-01-05,23.9,27.4,21.3,0.0,Región de Arica y Parinacota
...,...,...,...,...,...,...
37067,2023-12-28,6.7,10.2,4.2,1.7,Región de Magallanes y de la Antártica Chilena
37068,2023-12-29,8.7,11.1,4.7,2.3,Región de Magallanes y de la Antártica Chilena
37069,2023-12-30,10.8,15.1,7.7,0.0,Región de Magallanes y de la Antártica Chilena
37070,2023-12-31,10.7,13.6,8.3,1.0,Región de Magallanes y de la Antártica Chilena


### Agregación semanal

Las variables meteorológicas se agrupan utilizando el calendario ISO (año y semana ISO), calculando el promedio semanal para cada región.

Este criterio garantiza la compatibilidad temporal con el dataset agrícola.

In [214]:
df_meteo["time"] = pd.to_datetime(df_meteo["time"])

iso = df_meteo["time"].dt.isocalendar() # necesito usar ISO por que los datos agrícolas están por semana en este formato.
#ej: 1ra semana de 2020 parte del 30 de diciembre de 2019.

df_meteo["anio"] = iso.year
df_meteo["semana"] = iso.week

In [215]:
#Pasamos los datos diarios a semanales. las estadisticas pasarían de valores medios diarios a medias semanales
df_meteo_semanal = (
    df_meteo
    .groupby(["region", "anio", "semana"])
    .mean(numeric_only=True)
    .reset_index()
)

In [216]:
df_meteo_semanal

,region,anio,semana,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum
0,Región Metropolitana,2020,1,21.520000,28.900000,15.480000,0.000000
1,Región Metropolitana,2020,2,21.400000,29.542857,14.700000,0.000000
2,Región Metropolitana,2020,3,23.957143,32.400000,17.514286,0.000000
3,Región Metropolitana,2020,4,23.714286,31.771429,17.385714,0.000000
4,Región Metropolitana,2020,5,23.628571,31.585714,17.600000,0.000000
...,...,...,...,...,...,...,...
5305,Región del Maule,2026,23,10.928571,14.557143,8.242857,0.585714
5306,Región del Maule,2026,24,9.714286,13.028571,7.528571,3.442857
5307,Región del Maule,2026,25,8.828571,14.057143,5.200000,1.100000
5308,Región del Maule,2026,26,6.542857,12.585714,2.457143,0.042857


In [217]:
categorical_cols = list(df_meteo_semanal.dtypes[df_meteo_semanal.dtypes == 'str'].index) # solo region...


In [218]:
formatCategorical(df_meteo_semanal, categorical_cols)
formatColNames(df_meteo_semanal)

In [219]:
df_meteo_semanal

,region,anio,semana,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum
0,región_metropolitana,2020,1,21.520000,28.900000,15.480000,0.000000
1,región_metropolitana,2020,2,21.400000,29.542857,14.700000,0.000000
2,región_metropolitana,2020,3,23.957143,32.400000,17.514286,0.000000
3,región_metropolitana,2020,4,23.714286,31.771429,17.385714,0.000000
4,región_metropolitana,2020,5,23.628571,31.585714,17.600000,0.000000
...,...,...,...,...,...,...,...
5305,región_del_maule,2026,23,10.928571,14.557143,8.242857,0.585714
5306,región_del_maule,2026,24,9.714286,13.028571,7.528571,3.442857
5307,región_del_maule,2026,25,8.828571,14.057143,5.200000,1.100000
5308,región_del_maule,2026,26,6.542857,12.585714,2.457143,0.042857


In [220]:
print(df_agricola[["region", "anio", "semana"]].dtypes)
print(df_meteo_semanal[["region", "anio", "semana"]].dtypes)

region       str
anio      UInt32
semana    object
dtype: object
region       str
anio      UInt32
semana    UInt32
dtype: object


## Los datos agrícolas se limitan a 9 regiones :

        ['región_de_arica_y_parinacota',
         'región_de_coquimbo',
         'región_de_la_araucanía',
         'región_de_los_lagos',
         'región_de_ñuble',
         'región_de_valparaíso',
         'región_del_biobío',
         'región_del_maule',
         'región_metropolitana']
#### Los datos meteorológicos son para las 16 regiones. Hay que reducirlo y corregir discrepancias de nombres antes de realizar el Merge

In [174]:
list(df_agricola.region.unique())

In [221]:
df_meteo_semanal.region.unique()

<StringArray>
[                            'región_metropolitana',
                            'región_de_antofagasta',
                     'región_de_arica_y_parinacota',
                                'región_de_atacama',
                                  'región_de_aysén',
                               'región_de_coquimbo',
                           'región_de_la_araucanía',
                              'región_de_los_lagos',
                               'región_de_los_ríos',
   'región_de_magallanes_y_de_la_antártica_chilena',
                               'región_de_tarapacá',
                             'región_de_valparaíso',
                                  'región_de_ñuble',
                                'región_del_biobío',
 'región_del_libertador_general_bernardo_o'higgins',
                                 'región_del_maule']
Length: 16, dtype: str

In [222]:
#Diferencia en como se encuentra escrito la RM en ambos dataset
df_agricola.loc[df_agricola["region"] == "región_metropolitana_de_santiago", "region"] = "región_metropolitana"

### Nos quedamos solo con la data meteorológica de cuyas regiones tenemos información en nuestro dataset agrícola

In [223]:
regiones_agricolas = df_agricola["region"].unique()

df_meteo_semanal = df_meteo_semanal[
    df_meteo_semanal["region"].isin(regiones_agricolas)
]

In [224]:
df_meteo_semanal.region.unique()# cambios correctos

<StringArray>
[        'región_metropolitana', 'región_de_arica_y_parinacota',
           'región_de_coquimbo',       'región_de_la_araucanía',
          'región_de_los_lagos',         'región_de_valparaíso',
              'región_de_ñuble',            'región_del_biobío',
             'región_del_maule']
Length: 9, dtype: str

### Integración de fuentes

Finalmente, los datos agrícolas y meteorológicos se integran utilizando como claves:

- Región
- Año ISO
- Semana ISO


In [225]:
df_final = df_agricola.merge(
    df_meteo_semanal,
    on=["region", "anio", "semana"],
    how="left"
)

In [226]:
#un merge bien efectuad debería dar 0 na
df_final[df_final["temperature_2m_mean"].isna()]

,semana,fecha_inicio,region,tipo_punto_monitoreo,producto,variedad,calidad,unidad,precio_minimo,precio_maximo,precio_promedio,anio,mes,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum


In [227]:
#Comprobamos la eficacia del merge revisando si existen combinatorias que no fueron aplicadas/ darían NA
claves_agri = set(zip(df_agricola["region"],
                      df_agricola["anio"],
                      df_agricola["semana"]))

claves_meteo = set(zip(df_meteo_semanal["region"],
                   df_meteo_semanal["anio"],
                       df_meteo_semanal["semana"]))

faltantes = claves_agri - claves_meteo

len(faltantes)

0

In [228]:
df_final.head()

,semana,fecha_inicio,region,tipo_punto_monitoreo,producto,variedad,calidad,unidad,precio_minimo,precio_maximo,precio_promedio,anio,mes,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum
0,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,amarga,extra,$/kilo,3500.0,4000.0,3750.0,2020,12,23.4,26.78,20.7,0.62
1,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,amarga,primera,$/kilo,3000.0,3500.0,3250.0,2020,12,23.4,26.78,20.7,0.62
2,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,amarga,segunda,$/kilo,2500.0,3000.0,2750.0,2020,12,23.4,26.78,20.7,0.62
3,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,amarga,tercera,$/kilo,2000.0,2500.0,2250.0,2020,12,23.4,26.78,20.7,0.62
4,1,2019-12-30,región_de_arica_y_parinacota,mercado_minorista,aceituna,negra,extra,$/kilo,3000.0,3500.0,3250.0,2020,12,23.4,26.78,20.7,0.62


## Dataset final

El resultado corresponde a un dataset consolidado que contiene:

- información agrícola;
- variables meteorológicas agregadas semanalmente;
- fechas normalizadas;
- variables categóricas y nombres de columnas estandarizadas.

Este archivo constituye la base utilizada en las etapas posteriores de análisis exploratorio, entrenamiento y evaluación de modelos.

In [229]:
df_final.to_csv("./Data/processed_agricultural_prices.csv", index=False)